# Job Market Segmentation Pipeline

This notebook implements the full NLP pipeline used in the deployed
Job Intelligence Lab application.

Steps:
1. Load LinkedIn job posting dataset
2. Clean and normalize text
3. Generate sentence-transformer embeddings
4. Fuse title and description embeddings
5. Apply hierarchical clustering
6. Interpret job-market segments
7. Export deployment artifacts

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [ ]:
import sentence_transformers
import datasets
import sklearn
import torch
import matplotlib

print(matplotlib.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("datasets:", datasets.__version__)
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__)

In [ ]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
rng = np.random.default_rng(SEED)

ds = load_dataset("datastax/linkedin_job_listings") 

# Pick a split robustly
if isinstance(ds, dict) and "train" in ds:
    ds = ds["train"]
else:
    ds = ds[list(ds.keys())[0]] if isinstance(ds, dict) else ds

df = ds.to_pandas()
print("Columns:", list(df.columns)[:40])
print("Shape:", df.shape)

def clean_text(s):
    s = "" if s is None else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Find likely title + description columns
cols_lower = {c.lower(): c for c in df.columns}

def find_first_contains(keywords):
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in keywords):
            return c
    return None

col_title = find_first_contains(["title", "position", "job title", "role"])
col_desc  = find_first_contains(["description", "job description", "job_description", "desc", "summary"])

print("Detected title col:", col_title)
print("Detected desc  col:", col_desc)

# Build text column
parts = []
if col_title: parts.append(df[col_title].map(clean_text))
if col_desc:  parts.append(df[col_desc].map(clean_text))

if not parts:
    raise ValueError("No usable text columns found. Please show me df.columns and we’ll map it manually.")

df["text"] = parts[0]
for p in parts[1:]:
    df["text"] = df["text"] + " | " + p

# Remove too short / empty
df = df[df["text"].str.len() >= 80].reset_index(drop=True)

# Subsample for CPU-friendly hierarchical clustering
# Start with 5k
n = min(5000, len(df))                 
df = df.sample(n=n, random_state=SEED).reset_index(drop=True)

df.shape, df["text"].str.len().describe()

In [ ]:
import re
from typing import Tuple

df["title"] = df[col_title].fillna("").astype(str).str.strip()
df["desc_raw"] = df[col_desc].fillna("").astype(str).str.strip()

# Remove boilerplate patterns (EEO, benefits, privacy, accommodation, etc.)
BOILERPLATE_PATTERNS = [
    r"(?:^|\n)\s*equal opportunity\b.*",
    r"(?:^|\n)\s*eeo\b.*",
    r"(?:^|\n)\s*we are an equal opportunity\b.*",
    r"(?:^|\n)\s*reasonable accommodation\b.*",
    r"(?:^|\n)\s*accommodation\b.*",
    r"(?:^|\n)\s*background check\b.*",
    r"(?:^|\n)\s*drug[- ]?free\b.*",
    r"(?:^|\n)\s*benefits include\b.*",
    r"(?:^|\n)\s*benefits:\b.*",
    r"(?:^|\n)\s*privacy policy\b.*",
    r"(?:^|\n)\s*terms of use\b.*",
    r"(?:^|\n)\s*by applying\b.*you agree\b.*",
    r"(?:^|\n)\s*this is a volunteer opportunity\b.*",
    r"(?:^|\n)\s*in partnership with linkedin\b.*",
]

boiler_tail_re = re.compile("|".join(BOILERPLATE_PATTERNS), flags=re.IGNORECASE | re.DOTALL)

def clean_desc_tailcut(s: str, max_chars: int = 2000) -> Tuple[str, bool, bool]:
    s = "" if s is None else str(s)
    s = s.replace("\r\n", "\n").replace("\r", "\n").strip()

    m = boiler_tail_re.search(s)
    tailcut_hit = False
    if m:
        s = s[:m.start()].rstrip()
        tailcut_hit = True
    s = re.sub(r"\s+", " ", s.replace("\n", " ")).strip()

    s = re.sub(r"([a-z])([A-Z][a-z])", r"\1 \2", s)

    hard_trunc = len(s) > max_chars
    if hard_trunc:
        s = s[:max_chars].rstrip()

    return s, tailcut_hit, hard_trunc

out = df[col_desc].map(lambda x: clean_desc_tailcut(x, max_chars=2000))
df["desc_clean"]   = out.map(lambda t: t[0])
df["tailcut_hit"]  = out.map(lambda t: t[1])
df["hard_trunc"]   = out.map(lambda t: t[2])

print("tailcut_hit rate:", float(df["tailcut_hit"].mean()))
print("hard_trunc rate :", float(df["hard_trunc"].mean()))

In [ ]:
s = df["desc_clean"].fillna("").astype(str).str.lower()

flags = {
  "equal_opportunity": s.str.contains("equal opportunity"),
  "reasonable_accommodation": s.str.contains("reasonable accommodation"),
  "privacy_policy": s.str.contains("privacy policy"),
  "terms_of_use": s.str.contains("terms of use"),
  "linkedin_for_good": s.str.contains("linkedin for good"),
  "volunteermatch": s.str.contains("volunteermatch"),
}

for k, v in flags.items():
    print(k, float(v.mean()))

In [ ]:
raw = df["description"].fillna("").astype(str)
raw_nl = raw.str.replace("\r\n", "\n").str.replace("\r", "\n")
hit = raw_nl.map(lambda x: bool(boiler_tail_re.search(x)))
print("tailcut hit rate:", float(hit.mean()))

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

SEED = 42
ALPHA_MAIN = 0.75
BATCH_SIZE = 32
MODEL_NAME = "all-MiniLM-L6-v2"

embedder = SentenceTransformer(MODEL_NAME)

def _to_clean_str_list(x) -> list[str]:
    # Robust: handle None/NaN and non-str types
    return ["" if v is None else str(v) for v in x]

def embed_texts(texts, batch_size=BATCH_SIZE) -> np.ndarray:
    texts = _to_clean_str_list(texts)
    X = embedder.encode(
        texts,
        normalize_embeddings=False,  
        show_progress_bar=True,
        batch_size=batch_size,
    )
    X = np.asarray(X, dtype=np.float32)
    return X

def fuse_embeddings(X_title: np.ndarray, X_desc: np.ndarray, alpha: float) -> np.ndarray:
    if not (0.0 <= alpha <= 1.0):
        raise ValueError(f"alpha must be in [0, 1], got {alpha}")
    if X_title.shape != X_desc.shape:
        raise ValueError(f"shape mismatch: title {X_title.shape} vs desc {X_desc.shape}")

    # L2 normalize each source, then fuse + renormalize
    X_title_n = normalize(X_title, norm="l2")
    X_desc_n  = normalize(X_desc,  norm="l2")
    X = alpha * X_title_n + (1.0 - alpha) * X_desc_n
    X = normalize(X, norm="l2").astype(np.float32)

    # Make it contiguous (helps some downstream libs/perf)
    return np.ascontiguousarray(X)

# Compute embeddings
X_title = embed_texts(df["title"].tolist(), batch_size=BATCH_SIZE)
X_desc  = embed_texts(df["desc_clean"].tolist(), batch_size=BATCH_SIZE)

# Fuse
X_fused = fuse_embeddings(X_title, X_desc, alpha=ALPHA_MAIN)
print("X_title:", X_title.shape, X_title.dtype)
print("X_desc :", X_desc.shape,  X_desc.dtype)
print("X_fused:", X_fused.shape, X_fused.dtype, "contiguous:", X_fused.flags["C_CONTIGUOUS"])

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

def fit_proto_agglomerative(
    X, *,
    proto_count=300,
    final_k=10,
    linkage="complete",
    seed=42,
    batch_size=2048,
    n_init=5
):
    """
    Fit on X (train only).
    Returns artifacts needed to assign clusters to new points.
    Assumes X is already L2-normalized row-wise (recommended).
    """
    t0 = time.time()

    # Defensive: dtype + contiguous
    X = np.asarray(X, dtype=np.float32)
    if not X.flags["C_CONTIGUOUS"]:
        X = np.ascontiguousarray(X)

    # Prototype step (stochastic)
    mbk = MiniBatchKMeans(
        n_clusters=proto_count,
        random_state=seed,
        batch_size=batch_size,
        n_init=n_init
    )
    proto_ids = mbk.fit_predict(X)

    # Normalize prototypes (unit vectors)
    prototypes = normalize(mbk.cluster_centers_.astype(np.float32), norm="l2")
    prototypes = np.ascontiguousarray(prototypes)

    # Agglomerative step on prototypes
    agg = AgglomerativeClustering(
        n_clusters=final_k,
        linkage=linkage,
        metric="euclidean"
    )
    proto_cluster_labels = agg.fit_predict(prototypes)

    # Assign each training point to final cluster
    assigned = proto_cluster_labels[proto_ids]

    # Metrics (on prototype space)
    sil = silhouette_score(prototypes, proto_cluster_labels, metric="euclidean")
    max_cluster_pct = pd.Series(assigned).value_counts(normalize=True).max()
    runtime_sec = time.time() - t0

    artifacts = {
        "proto_count": proto_count,
        "final_k": final_k,
        "linkage": linkage,
        "seed": seed,
        "mbk": mbk,
        "prototypes": prototypes,                      # (proto_count, d), normalized
        "proto_cluster_labels": proto_cluster_labels,   # (proto_count,)
        "train_assigned": assigned,                     # (n_train,)
        "silhouette_proto": float(sil),
        "max_cluster_pct": float(max_cluster_pct),
        "runtime_sec": float(runtime_sec),
        "distance_metric": "euclidean",
    }
    return artifacts

def assign_with_artifacts(X_new, artifacts):
    """
    Assign new points using nearest prototype from trained mbk,
    then map prototype -> agglomerative cluster.
    Recommended: X_new is L2-normalized row-wise.
    """
    X_new = np.asarray(X_new, dtype=np.float32)
    if not X_new.flags["C_CONTIGUOUS"]:
        X_new = np.ascontiguousarray(X_new)
    
    proto_ids = artifacts["mbk"].predict(X_new)
    assigned = artifacts["proto_cluster_labels"][proto_ids]
    return assigned

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_FRAC = 0.8

idx = np.arange(len(df))
train_idx, test_idx = train_test_split(
    idx,
    test_size=(1 - TRAIN_FRAC),
    random_state=SEED,
    shuffle=True
)

# Defensive: ensure float32 + contiguous, avoids annoying downstream surprises
X_fused = np.asarray(X_fused, dtype=np.float32)
if not X_fused.flags["C_CONTIGUOUS"]:
    X_fused = np.ascontiguousarray(X_fused)

X_train = X_fused[train_idx]
X_test  = X_fused[test_idx]

df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_test  = df.iloc[test_idx].copy().reset_index(drop=True)

# Fit on train only
PROTO_MAIN = 300
K_MAIN = 10
LINKAGE_MAIN = "complete"

art = fit_proto_agglomerative(
    X_train,
    proto_count=PROTO_MAIN,
    final_k=K_MAIN,
    linkage=LINKAGE_MAIN,
    seed=SEED
)

# Assign test without refitting
df_train["cluster"] = art["train_assigned"]
df_test["cluster"]  = assign_with_artifacts(X_test, art)

print("Train silhouette (prototype space):", round(art["silhouette_proto"], 4))
print("Train max_cluster_pct:", round(art["max_cluster_pct"], 4))

print("\nTrain cluster sizes:\n",
      df_train["cluster"].value_counts().sort_values(ascending=False).head(10))

print("\nTest cluster sizes:\n",
      df_test["cluster"].value_counts().sort_values(ascending=False).head(10))

# Distribution shift summary
full_clusters = pd.Index(range(K_MAIN), name="cluster")

train_dist = df_train["cluster"].value_counts(normalize=True).reindex(full_clusters, fill_value=0.0)
test_dist  = df_test["cluster"].value_counts(normalize=True).reindex(full_clusters, fill_value=0.0)

dist_df = pd.DataFrame({"train_pct": train_dist, "test_pct": test_dist})
dist_df["abs_diff"] = (dist_df["train_pct"] - dist_df["test_pct"]).abs()

print("\nTrain vs Test cluster distribution (top diffs):\n",
      dist_df.sort_values("abs_diff", ascending=False).head(10))

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("train_idx head:", train_idx[:10])
print("test_idx head:", test_idx[:10])

Although the prototype-space silhouette score is modest (0.0337), this is expected for semantic embedding spaces where cluster boundaries are inherently soft. Importantly, cluster proportions are highly stable under an 80/20 train–test split. The maximum absolute difference in cluster prevalence is only 1.2%, indicating strong distributional stability and suggesting that the segmentation generalizes well rather than overfitting to the training sample.

In [ ]:
from scipy.spatial.distance import jensenshannon

js = jensenshannon(train_dist.values, test_dist.values)
print("JS divergence:", js)

JS divergence between train and test cluster distributions is low, further confirming stability.

In [ ]:
import time
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Sensitivvity Analysis
def run_sensitivity_alpha_linkage(
    X_title, X_desc,
    alphas=(0.6, 0.7, 0.75, 0.8),
    linkages=("average", "complete"),
    proto_count=300,
    final_k=10,
    seed=42
):
    rows = []

    for a in alphas:
        # Fuse
        X = fuse_embeddings(X_title, X_desc, alpha=a)

        # Fit prototype stage only once
        mbk = MiniBatchKMeans(
            n_clusters=proto_count,
            random_state=seed,
            batch_size=2048,
            n_init=5
        )
        proto_ids = mbk.fit_predict(X)
        prototypes = normalize(mbk.cluster_centers_.astype("float32"), norm="l2")

        for lk in linkages:
            t0 = time.time()

            agg = AgglomerativeClustering(
                n_clusters=final_k,
                linkage=lk,
                metric="euclidean"
            )
            proto_labels = agg.fit_predict(prototypes)
            assigned = proto_labels[proto_ids]

            sil = silhouette_score(
                prototypes,
                proto_labels,
                metric="euclidean"
            )
            max_pct = pd.Series(assigned).value_counts(normalize=True).max()
            rt = time.time() - t0

            rows.append({
                "alpha": a,
                "linkage": lk,
                "proto": proto_count,
                "k": final_k,
                "silhouette_proto": float(sil),
                "max_cluster_pct": float(max_pct),
                "runtime_sec_agglomerative_only": float(rt)
            })

    return pd.DataFrame(rows).sort_values("silhouette_proto", ascending=False)

sens_df = run_sensitivity_alpha_linkage(
    X_title, X_desc,
    alphas=(0.6, 0.7, 0.75, 0.8),
    linkages=("average", "complete"),
    proto_count=PROTO_MAIN,
    final_k=K_MAIN,
    seed=SEED
)

display(sens_df)

Sensitivity analysis over title–description weighting (α ∈ [0.6, 0.8]) and linkage strategy reveals that average linkage consistently produces a dominant cluster (max prevalence > 75%), indicating structural collapse. In contrast, complete linkage yields substantially more balanced partitions with comparable silhouette scores. We therefore adopt complete linkage with α = 0.75 for the main segmentation model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

def _safe_text(s) -> str:
    return "" if s is None else str(s)

def build_cluster_text(df: pd.DataFrame, title_col="title", desc_col="desc_clean") -> pd.Series:
    title = df[title_col].map(_safe_text).fillna("").astype(str).str.strip()
    desc  = df[desc_col].map(_safe_text).fillna("").astype(str).str.strip()
    text  = (title + " " + desc).str.replace(r"\s+", " ", regex=True).str.strip()
    return text

def top_tfidf_terms_by_cluster(
    df: pd.DataFrame,
    cluster_col: str,
    text_col: str,
    topn: int = 12,
    ngram_range=(1, 2),
    min_df: int = 5,
    max_df: float = 0.6,
    stop_words: str = "english",
):
    texts = df[text_col].fillna("").astype(str).tolist()

    vec = TfidfVectorizer(
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        stop_words=stop_words,
        lowercase=True,
    )
    X = vec.fit_transform(texts)  # (n_docs, n_terms)
    vocab = np.array(vec.get_feature_names_out())

    global_mean = np.asarray(X.mean(axis=0)).ravel()

    rows = []
    for cid, sub in df.groupby(cluster_col, sort=True):
        idx = sub.index.to_numpy()          # idx here are *row positions* because we reset_index later
        Xc = X[idx]
        c_mean = np.asarray(Xc.mean(axis=0)).ravel()

        delta = c_mean - global_mean
        top_idx = np.argsort(-delta)[:topn]

        rows.append({
            "cluster": int(cid),
            "n": int(len(sub)),
            "top_terms": ", ".join([vocab[j] for j in top_idx]),
            "top_terms_list": [vocab[j] for j in top_idx],
            "delta_scores": [float(delta[j]) for j in top_idx],
        })

    out = pd.DataFrame(rows).sort_values("n", ascending=False).reset_index(drop=True)
    return out, vec

def representative_examples_by_centroid(
    df: pd.DataFrame,
    X: np.ndarray,
    cluster_col: str,
    topk: int = 5,
    title_col="title",
    desc_col="desc_clean",
):
    # IMPORTANT: df index must be 0..n-1 and match X row order
    df = df.reset_index(drop=True)
    X = np.asarray(X, dtype=np.float32)
    if not X.flags["C_CONTIGUOUS"]:
        X = np.ascontiguousarray(X)

    assert len(df) == X.shape[0], f"df rows {len(df)} must align with X rows {X.shape[0]}"

    # make sure cosine sim is valid
    Xn = normalize(X, norm="l2")

    reps = []
    for cid, sub in df.groupby(cluster_col, sort=True):
        idx = sub.index.to_numpy()      
        Xc = Xn[idx]
        centroid = Xc.mean(axis=0, keepdims=True)
        centroid = normalize(centroid, norm="l2")

        sims = (Xc @ centroid.T).ravel()
        top_local = np.argsort(-sims)[:topk]
        top_pos = idx[top_local]

        for rank, pos in enumerate(top_pos, start=1):
            reps.append({
                "cluster": int(cid),
                "rank": rank,
                "cosine_to_centroid": float(Xn[pos] @ centroid.ravel()),
                "title": _safe_text(df.iloc[pos][title_col])[:180],
                "desc_snippet": _safe_text(df.iloc[pos][desc_col])[:220],
                "row_id": int(pos),
            })

    reps_df = pd.DataFrame(reps).sort_values(["cluster", "rank"]).reset_index(drop=True)
    return reps_df

def cluster_report(
    df: pd.DataFrame,
    X: np.ndarray,
    cluster_col="cluster",
    title_col="title",
    desc_col="desc_clean",
    top_terms_n=12,
    rep_k=5,
):
    # Force alignment: df order must match X order
    df = df.reset_index(drop=True).copy()

    if cluster_col not in df.columns:
        raise KeyError(f"'{cluster_col}' not found in df.columns")

    assert len(df) == X.shape[0], f"df rows {len(df)} != X rows {X.shape[0]}"

    size = df[cluster_col].value_counts().sort_values(ascending=False)
    size_df = size.rename_axis("cluster").reset_index(name="n")
    size_df["pct"] = size_df["n"] / size_df["n"].sum()

    df["text_for_labeling"] = build_cluster_text(df, title_col=title_col, desc_col=desc_col)

    kw_df, _ = top_tfidf_terms_by_cluster(
        df, cluster_col=cluster_col, text_col="text_for_labeling",
        topn=top_terms_n
    )

    reps_df = representative_examples_by_centroid(
        df, X, cluster_col=cluster_col,
        topk=rep_k, title_col=title_col, desc_col=desc_col
    )

    summary = (
        size_df.merge(kw_df[["cluster", "top_terms"]], on="cluster", how="left")
              .sort_values("n", ascending=False)
              .reset_index(drop=True)
    )

    return {
        "cluster_sizes": size_df,
        "keywords": kw_df,
        "representatives": reps_df,
        "summary": summary
    }

df_all = df.copy()
df_all["cluster"] = -1
df_all.loc[train_idx, "cluster"] = df_train["cluster"].values
df_all.loc[test_idx,  "cluster"] = df_test["cluster"].values

report = cluster_report(
    df=df_all,
    X=X_fused,
    cluster_col="cluster",
    title_col="title",
    desc_col="desc_clean",
    top_terms_n=12,
    rep_k=5,
)

print("=== Cluster summary (size + top terms) ===")
display(report["summary"])

print("\n=== Representatives (closest to centroid) ===")
display(report["representatives"].head(30))

In [ ]:
import re

def guess_cluster_name(top_terms: str, top_titles):
    s = (top_terms or "").lower()
    titles = " ".join(top_titles) if isinstance(top_titles, list) else str(top_titles or "")
    t = titles.lower()

    # very light rules 
    if any(k in s for k in ["nurse", "patient", "clinic", "rn", "pharmacy", "medical"]):
        return "Healthcare / Clinical"
    if any(k in s for k in ["software", "engineer", "cloud", "linux", "devops", "sql", "data"]):
        return "Tech / Software / Data"
    if any(k in s for k in ["sales", "customer", "account", "business development", "store"]):
        return "Sales / Customer / Retail"
    if any(k in s for k in ["maintenance", "equipment", "technician", "repair", "field service"]):
        return "Maintenance / Field Service"
    if any(k in s for k in ["manufacturing", "production", "operator", "warehouse", "cnc"]):
        return "Manufacturing / Operations"
    if any(k in s for k in ["project", "manager", "management", "operations manager"]):
        return "Management / PM / Ops"
    if any(k in s for k in ["laboratory", "lab", "research", "scientist"]):
        return "Lab / Research"
    if any(k in s for k in ["driver", "cdl", "delivery", "doordash", "dash"]):
        return "Delivery / Driver / Gig"
    if any(k in s for k in ["construction", "estimator", "electrical", "trade"]):
        return "Construction / Trades"
    return "Other / Mixed"

summary = report["summary"].copy()


if "top_titles" not in summary.columns:
    top_titles = (
        df_all.reset_index(drop=True)
             .groupby("cluster")["title"]
             .apply(lambda s: s.value_counts().head(3).index.tolist())
             .reset_index(name="top_titles")
    )
    summary = summary.merge(top_titles, on="cluster", how="left")

summary["cluster_name_guess"] = summary.apply(
    lambda r: guess_cluster_name(r.get("top_terms", ""), r.get("top_titles", [])),
    axis=1
)

display(summary[["cluster","n","pct","cluster_name_guess","top_terms","top_titles"]].sort_values("n", ascending=False))

In [ ]:
# Sanity check
reps = report["representatives"].copy()

def show_cluster_examples(cluster_id, k=3):
    sub = reps[reps["cluster"] == cluster_id].sort_values("rank").head(k)
    print(f"\n===== Cluster {cluster_id} | {summary.loc[summary.cluster==cluster_id,'cluster_name_guess'].values[0]} =====")
    for _, r in sub.iterrows():
        print(f"- ({r['cosine_to_centroid']:.3f}) {r['title']}")
        print(f"  {r['desc_snippet'][:180]}...")

for cid in summary.sort_values("n", ascending=False)["cluster"].tolist():
    show_cluster_examples(cid, k=3)

In [ ]:
# Stability Analysis
from sklearn.metrics import adjusted_rand_score

def fit_assign_full(X, proto_count=300, final_k=10, linkage="complete", seed=42):
    art_tmp = fit_proto_agglomerative(
        X,
        proto_count=proto_count,
        final_k=final_k,
        linkage=linkage,
        seed=seed
    )
    return art_tmp["train_assigned"], art_tmp

# Baseline
base_labels, base_art = fit_assign_full(
    X_fused,
    proto_count=PROTO_MAIN,
    final_k=K_MAIN,
    linkage=LINKAGE_MAIN,
    seed=SEED
)

seeds = [1, 7, 21, 42, 99]
rows = []
for sd in seeds:
    labels_sd, _ = fit_assign_full(
        X_fused,
        proto_count=PROTO_MAIN,
        final_k=K_MAIN,
        linkage=LINKAGE_MAIN,
        seed=sd
    )
    ari = adjusted_rand_score(base_labels, labels_sd)
    rows.append({"seed": sd, "ARI_vs_baseline": ari})

stab_df = pd.DataFrame(rows).sort_values("ARI_vs_baseline", ascending=False)
display(stab_df)
print("ARI mean:", stab_df["ARI_vs_baseline"].mean())

In [ ]:
import time
from sklearn.preprocessing import normalize
from sklearn.metrics import adjusted_rand_score

def fuse_embeddings(X_title, X_desc, alpha=0.75):
    """
    Fuse title/desc embeddings into one L2-normalized matrix.
    Assumes X_title, X_desc are (n,d). Output float32, contiguous.
    """
    Xt = np.asarray(X_title, dtype=np.float32)
    Xd = np.asarray(X_desc, dtype=np.float32)
    assert Xt.shape == Xd.shape, "X_title / X_desc shape mismatch"

    Xt = normalize(Xt, norm="l2")
    Xd = normalize(Xd, norm="l2")
    X  = alpha * Xt + (1.0 - alpha) * Xd
    X  = normalize(X, norm="l2").astype(np.float32)
    return np.ascontiguousarray(X)

def run_stability_grid(
    X_title, X_desc,
    *,
    alphas=(0.6, 0.7, 0.75, 0.8),
    linkages=("average", "complete"),
    proto_counts=(200, 300, 500),
    final_k=10,
    seeds=(42, 7, 21, 1, 99),
):
    """
    For each (alpha, linkage, proto_count), fit multiple seeds.
    Report:
      - ARI vs baseline seed (first seed in list)
      - silhouette on prototype space (from artifacts)
      - max_cluster_pct (degeneracy indicator)
      - runtime
    Note: ARI is permutation-invariant, so it's the right metric for stability.
    """
    rows = []

    for alpha in alphas:
        X = fuse_embeddings(X_title, X_desc, alpha=alpha)

        for linkage in linkages:
            for proto in proto_counts:

                labels_by_seed = {}
                sil_by_seed = {}
                maxpct_by_seed = {}
                rt_by_seed = {}

                # fit each seed
                for sd in seeds:
                    t0 = time.time()
                    art = fit_proto_agglomerative(
                        X,
                        proto_count=int(proto),
                        final_k=int(final_k),
                        linkage=str(linkage),
                        seed=int(sd),
                    )
                    rt_by_seed[sd] = time.time() - t0
                    labels_by_seed[sd] = np.asarray(art["train_assigned"])
                    sil_by_seed[sd] = float(art["silhouette_proto"])
                    maxpct_by_seed[sd] = float(art["max_cluster_pct"])

                # baseline = first seed
                base_seed = seeds[0]
                base_labels = labels_by_seed[base_seed]

                ari_list = []
                for sd in seeds:
                    ari_list.append(adjusted_rand_score(base_labels, labels_by_seed[sd]))

                rows.append({
                    "alpha": float(alpha),
                    "linkage": str(linkage),
                    "proto": int(proto),
                    "k": int(final_k),

                    "ARI_mean_vs_base": float(np.mean(ari_list)),
                    "ARI_min_vs_base": float(np.min(ari_list)),
                    "ARI_std_vs_base": float(np.std(ari_list)),

                    "silhouette_proto_mean": float(np.mean(list(sil_by_seed.values()))),
                    "max_cluster_pct_mean": float(np.mean(list(maxpct_by_seed.values()))),
                    "runtime_sec_mean": float(np.mean(list(rt_by_seed.values()))),

                    "base_seed": int(base_seed),
                    "seeds": list(seeds),
                })

    df_grid = pd.DataFrame(rows)

    # A practical ranking rule:
    # 1) stability first (ARI_mean, then ARI_min)
    # 2) separation second (silhouette)
    # 3) avoid degenerate clusters (max_cluster_pct low)
    df_grid = df_grid.sort_values(
        ["ARI_mean_vs_base", "ARI_min_vs_base", "silhouette_proto_mean", "max_cluster_pct_mean"],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

    return df_grid

K_MAIN = K_MAIN if "K_MAIN" in globals() else 10

grid_df = run_stability_grid(
    X_title, X_desc,
    alphas=(0.6, 0.7, 0.75, 0.8),
    linkages=("average", "complete"),
    proto_counts=(200, 300, 500),
    final_k=K_MAIN,
    seeds=(42, 7, 21, 1, 99),
)

print("Top 12 configs (sorted by stability -> separation -> degeneracy):")
display(grid_df.head(12))

best = grid_df.iloc[0].to_dict()
print("\nSelected (best) config:")
print(best)

# lock in the final model on full data:
ALPHA_FINAL   = best["alpha"]
LINKAGE_FINAL = best["linkage"]
PROTO_FINAL   = best["proto"]
SEED_FINAL    = best["base_seed"]  # baseline seed (42)

X_fused_final = fuse_embeddings(X_title, X_desc, alpha=ALPHA_FINAL)
art_final = fit_proto_agglomerative(
    X_fused_final,
    proto_count=PROTO_FINAL,
    final_k=K_MAIN,
    linkage=LINKAGE_FINAL,
    seed=SEED_FINAL
)
df["cluster"] = art_final["train_assigned"]

print("\nFinal model quick stats:")
print("alpha:", ALPHA_FINAL, "| linkage:", LINKAGE_FINAL, "| proto:", PROTO_FINAL, "| k:", K_MAIN, "| seed:", SEED_FINAL)
print("silhouette (proto):", round(art_final["silhouette_proto"], 4))
print("max_cluster_pct:", round(art_final["max_cluster_pct"], 4))
print(df["cluster"].value_counts().head(10))

In [ ]:
from sklearn.metrics import adjusted_rand_score
import numpy as np
import pandas as pd

def run_k_sensitivity(
    X_title, X_desc,
    *,
    alpha=0.75,
    linkage="average",
    proto=200,
    ks=(10, 12, 15),
    seeds=(42, 7, 21),
):
    rows = []

    # fuse once
    X = fuse_embeddings(X_title, X_desc, alpha=alpha)

    for k in ks:

        labels_by_seed = {}
        sil_by_seed = {}
        maxpct_by_seed = {}

        for sd in seeds:
            art = fit_proto_agglomerative(
                X,
                proto_count=proto,
                final_k=k,
                linkage=linkage,
                seed=sd,
            )

            labels_by_seed[sd] = np.asarray(art["train_assigned"])
            sil_by_seed[sd] = float(art["silhouette_proto"])
            maxpct_by_seed[sd] = float(art["max_cluster_pct"])

        base_seed = seeds[0]
        base_labels = labels_by_seed[base_seed]

        ari_list = [
            adjusted_rand_score(base_labels, labels_by_seed[sd])
            for sd in seeds
        ]

        rows.append({
            "k": k,
            "ARI_mean": float(np.mean(ari_list)),
            "ARI_min": float(np.min(ari_list)),
            "ARI_std": float(np.std(ari_list)),
            "silhouette_proto_mean": float(np.mean(list(sil_by_seed.values()))),
            "max_cluster_pct_mean": float(np.mean(list(maxpct_by_seed.values()))),
        })

    return pd.DataFrame(rows).sort_values("k").reset_index(drop=True)


k_df = run_k_sensitivity(
    X_title, X_desc,
    alpha=0.75,
    linkage="average",
    proto=200,
    ks=(10, 12, 15),
    seeds=(42, 7, 21),
)

display(k_df)

In [ ]:
test_df = run_k_sensitivity(
    X_title, X_desc,
    alpha=0.6,
    linkage="average",
    proto=200,
    ks=(12,),
    seeds=(42,7,21),
)

test_df2 = run_k_sensitivity(
    X_title, X_desc,
    alpha=0.5,
    linkage="average",
    proto=200,
    ks=(12,),
    seeds=(42,7,21),
)

display(pd.concat([test_df.assign(alpha=0.6),
                   test_df2.assign(alpha=0.5)], ignore_index=True))

In [ ]:
desc_only_df = run_k_sensitivity(
    X_title, X_desc,
    alpha=0.0,
    linkage="average",
    proto=200,
    ks=(12,),
    seeds=(42,7,21),
)

display(desc_only_df.assign(alpha=0.0))

In [ ]:
import numpy as np
import pandas as pd

CLUSTER_COL_L1 = "cluster"

# For interpretability, we will do a second-level clustering within the dominant cluster from level 1
TECH_C = 0
vc = df_all[CLUSTER_COL_L1].value_counts()
tech_n = int(vc.loc[TECH_C])
tech_pct = float(tech_n / len(df_all))

print(f"[Level-1] Tech cluster = {TECH_C}, n={tech_n}, pct={tech_pct:.3f}")

# Get subset from the dominant cluster
tech_mask = (df_all[CLUSTER_COL_L1].values == TECH_C)
tech_idx  = np.where(tech_mask)[0]

X_tech = X_fused[tech_idx]
df_tech = df_all.iloc[tech_idx].copy()

print("X_tech:", X_tech.shape, "df_tech:", df_tech.shape)

# Level-2 clustering
PROTO_L2_TECH = int(min(200, max(50, X_tech.shape[0] // 20)))
K_L2_TECH = 8
LINKAGE_L2_TECH = "complete"
SEED_L2_TECH = 42

art_tech_l2 = fit_proto_agglomerative(
    X_tech,
    proto_count=PROTO_L2_TECH,
    final_k=K_L2_TECH,
    linkage=LINKAGE_L2_TECH,
    seed=SEED_L2_TECH
)

df_tech["cluster_l2_tech"] = art_tech_l2["train_assigned"]

print("[Tech L2] silhouette_proto:", round(art_tech_l2["silhouette_proto"], 4))
print("[Tech L2] max_cluster_pct:", round(art_tech_l2["max_cluster_pct"], 4))
print(df_tech["cluster_l2_tech"].value_counts().head(10))

In [ ]:
report_l2_tech = cluster_report(
    df=df_tech,
    X=X_tech,
    cluster_col="cluster_l2_tech",
    top_terms_n=12,
    rep_k=5
)

display(report_l2_tech["summary"])
display(report_l2_tech["representatives"].head(30))

In [ ]:
# Also refine the second large cluster in Lelvel-1 Clustering
SALES_C = 3

sales_mask = (df_all[CLUSTER_COL_L1].values == SALES_C)
sales_idx  = np.where(sales_mask)[0]

X_sales = X_fused[sales_idx]
df_sales = df_all.iloc[sales_idx].copy()

print("X_sales:", X_sales.shape, "df_sales:", df_sales.shape)

In [ ]:
PROTO_L2_SALES = int(min(200, max(50, X_sales.shape[0] // 20)))
K_L2_SALES = 6
LINKAGE_L2_SALES = "complete"
SEED_L2_SALES = 42

art_sales_l2 = fit_proto_agglomerative(
    X_sales,
    proto_count=PROTO_L2_SALES,
    final_k=K_L2_SALES,
    linkage=LINKAGE_L2_SALES,
    seed=SEED_L2_SALES
)

df_sales["cluster_l2_sales"] = art_sales_l2["train_assigned"]

print("silhouette:", round(art_sales_l2["silhouette_proto"], 4))
print("max_cluster_pct:", round(art_sales_l2["max_cluster_pct"], 4))
print(df_sales["cluster_l2_sales"].value_counts())

In [ ]:
report_l2_sales = cluster_report(
    df=df_sales,
    X=X_sales,
    cluster_col="cluster_l2_sales",
    top_terms_n=12,
    rep_k=5
)

display(report_l2_sales["summary"])
display(report_l2_sales["representatives"].head(30))

In [ ]:
import numpy as np
import pandas as pd

df_all = df_all.copy()

# L1 cluster already exists: df_all["cluster"]
df_all["cluster_l2"] = np.nan         
df_all["cluster_path"] = ""           
df_all["segment_name"] = ""         

# merge Tech L2 back
df_all.loc[df_tech.index, "cluster_l2"] = df_tech["cluster_l2_tech"].values
df_all.loc[df_tech.index, "cluster_path"] = (
    "L1-" + df_all.loc[df_tech.index, "cluster"].astype(int).astype(str)
    + " > Tech.L2-" + df_tech["cluster_l2_tech"].astype(int).astype(str)
)

# merge Sales L2 back
df_all.loc[df_sales.index, "cluster_l2"] = df_sales["cluster_l2_sales"].values
df_all.loc[df_sales.index, "cluster_path"] = (
    "L1-" + df_all.loc[df_sales.index, "cluster"].astype(int).astype(str)
    + " > Sales.L2-" + df_sales["cluster_l2_sales"].astype(int).astype(str)
)

# Fill non-refined rows (only Level-1)
mask_no_l2 = df_all["cluster_l2"].isna()

df_all.loc[mask_no_l2, "cluster_path"] = (
    "L1-" + df_all.loc[mask_no_l2, "cluster"].astype(int).astype(str)
)

# sanity checks
print("cluster_path examples:")
display(df_all[["cluster", "cluster_l2", "cluster_path"]].dropna(subset=["cluster_path"]).head(10))

print("\nHow many got refined to L2?")
print(df_all["cluster_l2"].notna().mean())

print("\nCounts by cluster_path (top 20):")
display(df_all["cluster_path"].value_counts().head(20))

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

# --------- helpers ----------
def _tokenize(s: str):
    s = (s or "").lower()
    s = re.sub(r"[^a-z0-9\s\-/&]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.split()

def _top_titles(df_seg: pd.DataFrame, k=8, title_col="title"):
    # get top frequent job titles
    titles = df_seg[title_col].fillna("").astype(str).str.lower().tolist()
    # light normalization
    titles = [re.sub(r"\s+", " ", t).strip() for t in titles]
    cnt = Counter([t for t in titles if t])
    return [t for t,_ in cnt.most_common(k)]

# keyword->label mapping
LABEL_RULES = [
    ("Software Engineering", ["software", "developer", "backend", "frontend", "full stack", "java", "python", "cloud", "azure", "aws", "devops", "application", "platform engineer", "software engineer"]),
    ("Data / Analytics", ["data", "analyst", "analytics", "bi", "sql", "tableau", "power bi", "data warehouse", "etl", "modeling", "machine learning"]),
    ("IT / Security", ["security", "cyber", "infosec", "information security", "soc", "siem", "network", "iam", "risk"]),
    ("Customer Service / Call Center", ["customer service", "csr", "call center", "customer care", "support", "inbound", "outbound"]),
    ("Sales (General)", ["sales", "account", "bd", "business development", "quota", "territory", "pipeline"]),
    ("Retail / Store Ops", ["store", "retail", "cashier", "shift", "associate", "merchandise"]),
    ("Project / Program Management", ["project manager", "program", "pmp", "schedule", "stakeholder", "delivery", "scrum"]),
    ("HR / Admin", ["hr", "human resources", "recruit", "payroll", "administrative", "coordinator", "assistant"]),
    ("Healthcare (Clinical)", ["patient", "nurse", "rn", "clinic", "clinical", "therapy", "therapist", "medical", "phlebotomist"]),
    ("Healthcare (Pharmacy/Lab)", ["pharmacy", "pharmacist", "laboratory", "lab", "technician"]),
    ("Field Service / Maintenance", ["technician", "maintenance", "repair", "equipment", "field service"]),
    ("Food Service / Hospitality", ["restaurant", "server", "food", "dining", "kitchen", "cook"]),
    ("Construction / Estimating", ["construction", "estimator", "bids", "project engineer", "site"]),
    ("Logistics / Warehouse", ["warehouse", "fulfillment", "picker", "pack", "inventory", "forklift"]),
]

def suggest_segment_name(top_terms: str, top_titles: list, l1_hint: str = ""):
    # combine evidence text
    evidence = " ".join([top_terms or "", " ".join(top_titles or []), l1_hint or ""]).lower()

    hits = []
    for label, keys in LABEL_RULES:
        score = 0
        for k in keys:
            if k in evidence:
                score += 1
        if score > 0:
            hits.append((label, score))

    if not hits:
        return "Other / Mixed", 0

    hits.sort(key=lambda x: (-x[1], x[0]))
    best_label, best_score = hits[0]
    return best_label, int(best_score)

# Build segment summary table
def build_segment_table(df_all: pd.DataFrame, path_col="cluster_path", title_col="title", text_col_for_terms="desc_clean"):
    rows = []
    for path, seg in df_all.groupby(path_col):
        n = len(seg)
        pct = n / len(df_all)

        # Here we'll do a light heuristic: most common tokens from titles+desc (NOT perfect, but good for naming).
        tokens = []
        for s in (seg[title_col].fillna("").astype(str).tolist() + seg[text_col_for_terms].fillna("").astype(str).tolist()):
            tokens += _tokenize(s)
        stop = set(["the","and","to","of","in","a","for","with","on","at","is","are","as","an","or","be","will","you","we"])
        tokens = [t for t in tokens if (t not in stop and len(t) >= 3 and not t.isdigit())]
        top_terms = ", ".join([t for t,_ in Counter(tokens).most_common(12)])

        top_titles = _top_titles(seg, k=8, title_col=title_col)

        # optional L1 hint from path ("Tech" / "Sales")
        l1_hint = ""
        if "Tech" in path: l1_hint = "tech software data"
        if "Sales" in path: l1_hint = "sales customer retail"

        name, conf = suggest_segment_name(top_terms, top_titles, l1_hint=l1_hint)

        rows.append({
            "cluster_path": path,
            "n": n,
            "pct": pct,
            "segment_name_suggested": name,
            "name_confidence": conf,
            "top_terms": top_terms,
            "top_titles": top_titles[:5],
        })

    out = pd.DataFrame(rows).sort_values(["n"], ascending=False).reset_index(drop=True)
    return out

segment_table = build_segment_table(df_all, path_col="cluster_path", title_col="title", text_col_for_terms="desc_clean")
display(segment_table.head(30))

In [ ]:
# Adjustment for final naming manually
FINAL_NAME_MAP = {
    # Sales / business
    "L1-1":               "Sales – Account / Enterprise",

    # Healthcare
    "L1-5":               "Healthcare – Clinical",
    "L1-9":               "Healthcare – Pharmacy / Lab",
    "L1-0 > Tech.L2-1":  "Healthcare Admin (Non-Clinical)",
    "L1-0 > Tech.L2-2":  "Clinical Therapy",

    # Operations / logistics
    "L1-4":               "Logistics / Warehouse",
    "L1-6":               "Field Service / Maintenance",

    # Construction
    "L1-8":               "Construction / Estimating",

    # Tech segments
    "L1-0 > Tech.L2-0":  "Data & Business Analytics",
    "L1-0 > Tech.L2-4":  "Software Engineering",
    "L1-0 > Tech.L2-3":  "Cyber / Information Security",
    "L1-0 > Tech.L2-6":  "Education / Teaching",
    "L1-0 > Tech.L2-7":  "Bilingual / Community Support",

    # Food / Hospitality — L1-2 and L1-0 > Tech.L2-5 consolidated
    "L1-2":               "Food Service / Hospitality",
    "L1-0 > Tech.L2-5":   "Food Service / Hospitality", 

    # Retail / service
    "L1-3 > Sales.L2-5": "Customer Service / Retail",
    "L1-3 > Sales.L2-0": "Billing / Client Operations",
    "L1-3 > Sales.L2-1": "Administrative / Executive Support",
    "L1-3 > Sales.L2-2": "Hospitality / Food Service",
    "L1-3 > Sales.L2-4": "Legal Support",

    # Management
    "L1-3 > Sales.L2-3": "Project / Program Management",

    # Platform work
    "L1-7":               "Gig / Delivery (Platform Work)",
}

The hierarchical pipeline produced 22 structural cluster paths, but after semantic review two clusters were merged into the same business category (“Food Service / Hospitality”), resulting in 21 final interpretable job segments.

In [ ]:
# Apply final segment names
df_all["segment_name_final"] = df_all["cluster_path"].map(FINAL_NAME_MAP)

# Sanity check
app_segments = [
    "Sales – Account / Enterprise",
    "Healthcare – Clinical",
    "Logistics / Warehouse",
    "Customer Service / Retail",
    "Data & Business Analytics",
    "Software Engineering",
    "Field Service / Maintenance",
    "Project / Program Management",
    "Billing / Client Operations",
    "Healthcare Admin (Non-Clinical)",
    "Food Service / Hospitality",
    "Clinical Therapy",
    "Administrative / Executive Support",
    "Hospitality / Food Service",
    "Healthcare – Pharmacy / Lab",
    "Cyber / Information Security",
    "Legal Support",
    "Education / Teaching",
    "Construction / Estimating",
    "Bilingual / Community Support",
    "Gig / Delivery (Platform Work)",
]
actual_segments = set(df_all["segment_name_final"].dropna().unique())
print("=== Segment name alignment check ===")
all_ok = True
for s in app_segments:
    match = s in actual_segments
    if not match:
        all_ok = False
    print(f"  {'✓' if match else '✗ MISMATCH'} | {s}")

missing = df_all["segment_name_final"].isna().sum()
print(f"\nMissing segment_name_final mappings: {missing}")
print(f"Total segments: {df_all['segment_name_final'].nunique()}")
print(f"All checks passed: {all_ok and missing == 0}")

In [ ]:
# Segment stats table (verify names match app exactly)
segment_table["segment_name_final"] = segment_table["cluster_path"].map(FINAL_NAME_MAP)
display(segment_table[[
    "cluster_path",
    "n",
    "pct",
    "segment_name_suggested",
    "segment_name_final",
    "name_confidence",
]].head(30))

In [ ]:
# Market insights
SAL_CAP = 300_000  

SAL_COL = "normalized_salary"

seg_stats = (
    df_all
    .dropna(subset=["segment_name_final"])
    .assign(sal_clean=lambda d: d[SAL_COL].where(d[SAL_COL] <= SAL_CAP)) 
    .groupby("segment_name_final")
    .agg(
        n_jobs=("job_id", "count"),
        avg_salary=("sal_clean", "mean"),
        med_salary=("sal_clean", "median"),
    )
    .sort_values("n_jobs", ascending=False)
    .reset_index()
)
display(seg_stats)

In [ ]:
# Segment centroids
from sklearn.preprocessing import normalize
import numpy as np

valid_idx = df_all.index[df_all["segment_name_final"].notna()].to_numpy()

X   = X_fused[valid_idx]
seg = df_all.loc[valid_idx, "segment_name_final"].to_numpy()

segment_centroids = {}
for s in np.unique(seg):
    vecs = X[seg == s]
    c    = vecs.mean(axis=0, keepdims=True)
    c    = normalize(c, norm="l2")[0]
    segment_centroids[s] = c

print(f"Segments: {len(segment_centroids)}")
print("Keys:", sorted(segment_centroids.keys()))

In [ ]:
# Save artifacts
import os

os.makedirs("artifacts", exist_ok=True)

df_all.to_parquet("artifacts/df_all.parquet", index=False)
np.save("artifacts/X_fused.npy", X_fused)
np.savez(
    "artifacts/segment_centroids.npz",
    **{k: v for k, v in segment_centroids.items()}
)

print("Artifacts saved.")
print(f"  df_all:             {df_all.shape}")
print(f"  X_fused:            {X_fused.shape}")
print(f"  segment_centroids:  {len(segment_centroids)} segments")